# 99 · Orquestrador: executa o pipeline de ponta a ponta

Este notebook **encadeia todas as etapas** numa única execução, com um único dicionário de configuração (`cfg`) e uma única execução no MLflow. Os notebooks 00 a 06 só definem funções; é aqui que elas rodam em ordem.

| etapa | notebook | função | entra | sai (principal) |
|---|---|---|---|---|
| 1 | 01_entrada | `executa_etapa_entrada` | tabela de entrada (Spark) | base validada, com a coluna `particao` (treino, teste, oot, hist) |
| 2–3 | 02_selecao | `executa_etapa_selecao` | base da etapa 1 | variáveis selecionadas (pré-filtros + Boruta) |
| 4 | 03_automl | `executa_etapa_automl` | saída da etapa 2 | modelo vencedor e `score_bruto` |
| 5–6 | 04_calibracao_explicabilidade | `executa_etapa_calibracao_explicabilidade` | saídas das etapas 2 e 4 | `score_calibrado`, SHAP e importância |
| 7 | 05_regua | `executa_etapa_regua` | base calibrada | régua (ratings), backtest por safra e por período |
| 8 | 06_lifetime | `executa_etapa_lifetime` | base com rating e régua | curvas de PD lifetime por rating (**só PD**) |

**Fluxo de dados.** Tudo fica em Spark até a régua; o pandas entra só onde a tabela já é pequena (agregados por faixa, safra ou idade) e nas duas amostras do desenho: a do Boruta (etapa 3) e a da explicabilidade (etapa 6).

**Rastreabilidade.** Toda execução recebe um `id_execucao`. Ele vai em todas as tabelas Delta gravadas (`<esquema>.<prefixo>_<tarefa>_<nome>`), no nome da execução do MLflow e no resumo final. Com ele, qualquer número de qualquer tabela pode ser ligado à configuração exata que o produziu (gravada em `configuracao.json` no MLflow).

**Como usar.**

1. Preencha a célula de **parâmetros da execução** (seção 2).
2. Rode o notebook inteiro (*Run all*).
3. Consulte os resultados na seção 4, nas tabelas Delta do esquema de saída e na execução do MLflow.

## 1. Carrega as funções de todas as etapas

O comando `%run` executa outro notebook **dentro desta sessão**: as funções definidas lá passam a existir aqui. No Databricks, cada `%run` precisa estar sozinho na sua célula.

O `04_calibracao_explicabilidade` já executa o `03_automl` (ele usa funções do 03 para preparar a amostra do espelho), por isso o 03 não aparece abaixo. O `00_config` é executado de novo dentro de cada notebook; isso não tem efeito colateral, porque ele só define tabelas e funções.

In [ ]:
%run ./00_config

In [ ]:
%run ./01_entrada

In [ ]:
%run ./02_selecao

In [ ]:
%run ./04_calibracao_explicabilidade

In [ ]:
%run ./05_regua

In [ ]:
%run ./06_lifetime

## 2. Parâmetros da execução

Estas são as **únicas** células que o usuário precisa editar. O que não aparece aqui usa o padrão do `00_config` (ver `mostra_configuracao(cfg)` abaixo); qualquer padrão pode ser trocado em `AJUSTES`, com a mesma estrutura de seções do `00_config`.

**Escolhas principais**

| parâmetro | PD | LGD |
|---|---|---|
| `TAREFA` | `"pd"` | `"lgd"` |
| `METRICA` (decide a vencedora no OOT) | `ks`, `auc`, `gini`, `log_loss`, `brier` | `rmse`, `mae`, `r2`, `vies`, `spearman` |
| `TECNICAS` | `logistica`, `random_forest`, `gbt`, `lightgbm`, `xgboost` | `logistica_fracionaria`, `random_forest`, `gbt`, `lightgbm`, `xgboost`, `duas_etapas` |
| `DATA_CORTE` | último mês com desempenho observado; define quais safras estão maduras (12 meses de desempenho) | idem, com a janela de cobrança de 24 meses |

**Contrato da tabela de entrada** (só colunas essenciais; detalhes no notebook 01):

| PD | LGD |
|---|---|
| `id_contrato`, `safra`, `amostra` (DES/OOT/HIST), `alvo_12m` (0/1), `data_primeiro_default`, `data_fim_observacao` + variáveis explicativas | `id_contrato`, `safra_default`, `amostra` (DES/OOT), `lgd_realizada` (entre 0 e 1) + variáveis explicativas |

In [ ]:
# -----------------------------------------------------------------------------
# PARÂMETROS DA EXECUÇÃO: edite aqui
# -----------------------------------------------------------------------------
TABELA_ENTRADA = "catalogo.risco.base_desafiante_pd"   # tabela Delta no contrato do notebook 01
TAREFA = "pd"                                           # "pd" ou "lgd"
METRICA = "ks"                                          # métrica que escolhe a técnica vencedora no OOT
DATA_CORTE = "2025-09-01"                               # data de referência da maturidade das safras
ESQUEMA_SAIDA = "catalogo.risco_desafiante"             # onde as tabelas de resultado são gravadas
EXPERIMENTO_MLFLOW = "/Users/voce@banco.com/desafiante_pd"   # None = não registra no MLflow
GRAVAR = True                                           # False = só roda e exibe (nada é gravado)

# Técnicas que o AutoML compara (None = todas as da tarefa).
TECNICAS = None

# Ajustes de qualquer padrão do 00_config, na mesma estrutura de seções.
# Exemplos (descomente para usar):
AJUSTES = {
    # "caminho_artefatos": "/Volumes/catalogo/risco/modelos",   # salva o modelo ajustado
    # "regua": {"n_faixas": "auto"},                             # nº de ratings escolhido pelos dados
    # "automl": {"usar_dois_finalistas": True},                  # otimiza os dois primeiros se empatarem
    # "calibracao": {"metodo": "isotonica"},
}

### Variante LGD

Para rodar a LGD, troque a célula acima por algo como:

```python
TABELA_ENTRADA = "catalogo.risco.base_desafiante_lgd"
TAREFA = "lgd"
METRICA = "rmse"
DATA_CORTE = "2025-09-01"
ESQUEMA_SAIDA = "catalogo.risco_desafiante"
EXPERIMENTO_MLFLOW = "/Users/voce@banco.com/desafiante_lgd"
GRAVAR = True
TECNICAS = None          # inclui o modelo de duas etapas
AJUSTES = {
    # duas etapas com XGBoost nas duas partes, em vez do LightGBM padrão:
    # "automl": {"padrao_referencia": {"duas_etapas": {"tecnica_etapa1": "xgboost", "tecnica_etapa2": "xgboost"}}},
}
```

Na LGD a etapa 8 (lifetime) não roda: a LGD não tem curva por idade do contrato. O resto do fluxo é o mesmo, com as métricas e os testes próprios da LGD (calibração linear, teste de Welch na fusão de ratings, intervalo t no backtest).

In [ ]:
# Monta e valida a configuração. Erros de digitação em AJUSTES (uma chave que
# não existe no padrão) param aqui, antes de qualquer processamento.
ajustes_execucao = dict(AJUSTES)
if TECNICAS is not None:
    ajustes_execucao["tecnicas"] = TECNICAS

cfg = cria_configuracao(tarefa=TAREFA, metrica=METRICA, data_corte=DATA_CORTE,
                        esquema_saida=ESQUEMA_SAIDA if GRAVAR else None,
                        experimento_mlflow=EXPERIMENTO_MLFLOW, ajustes=ajustes_execucao)
mostra_configuracao(cfg)

## 3. Execução

A função `executa_pipeline` roda as etapas em ordem e mede o tempo de cada uma.

**MLflow.** Se houver experimento configurado, tudo acontece dentro de **uma** execução (*run*) do MLflow, chamada `<tarefa>_<id_execucao>`. As etapas registram nela os seus gráficos (`registra_figura_mlflow`), e o orquestrador registra a configuração completa, as versões das bibliotecas, as etiquetas da execução (técnica e versão finais) e o resumo; as métricas do modelo final no teste e no OOT (`final_<métrica>_<conjunto>`) vêm do notebook 03. Se uma etapa falhar, a execução do MLflow é encerrada com status de falha, para não ficar "aberta" na interface.

**Resumo da execução.** Ao final, uma tabela de uma linha por etapa diz o tempo gasto e o resultado principal (ex.: "logistica (otimizado): ks OOT = 0,412"). Ela é gravada em `<prefixo>_<tarefa>_resumo_execucao`.

In [ ]:
import time

import pandas as pd


def _registra_etiquetas_mlflow(etapa_automl, cfg):
    """
    Registra etiquetas (tags) da execução no MLflow: técnica e versão finais,
    tarefa e id_execucao. Com elas, a lista de execuções do experimento pode
    ser filtrada (ex.: todas as execuções em que o XGBoost venceu).

    As métricas do modelo final (final_<métrica>_<conjunto>) já são
    registradas pelo próprio notebook 03.
    """
    import mlflow
    mlflow.set_tags({"tecnica_final": etapa_automl["tecnica_final"],
                     "versao_final": etapa_automl["versao_final"],
                     "tarefa": cfg["tarefa"], "id_execucao": cfg["id_execucao"]})


def _resumo_regua(etapa_regua, cfg):
    """Texto curto com o número de ratings e a situação do backtest no OOT."""
    backtest = etapa_regua["backtest"]
    por_periodo = backtest["rating_periodo"]
    oot = por_periodo[por_periodo["particao"] == "oot"]
    alertas = int((oot["status"] == "alerta").sum())
    recalibrar = sorted(backtest["rating_safra"].loc[backtest["rating_safra"]["recalibrar"], "rating"].unique())
    texto = (f"{etapa_regua['n_faixas_iniciais']} faixas iniciais → {len(etapa_regua['regua'])} ratings; "
             f"OOT: {alertas} rating(s) em alerta")
    if recalibrar:
        texto += f"; recalibrar (alerta persistente): {', '.join(map(str, recalibrar))}"
    return texto


def _resumo_lifetime(etapa_lifetime):
    """Texto curto com o veredito do backtest da curva (OOT e HIST)."""
    backtest = etapa_lifetime["backtest"]
    com_dados = backtest[backtest["veredito"] != "sem dados"]
    dentro = int((com_dados["veredito"] == "dentro do IC").sum())
    return (f"{len(etapa_lifetime['curvas'])} curvas; backtest: {dentro} de {len(com_dados)} "
            f"pontos (rating × fonte × horizonte) dentro do IC 95%")


def executa_pipeline(spark, sdf_entrada, cfg, gravar=True):
    """
    Executa as etapas 1 a 8 em ordem e devolve um dicionário com a saída de cada uma.

    Parâmetros
        spark        sessão Spark.
        sdf_entrada  DataFrame Spark no contrato de entrada do notebook 01.
        cfg          configuração criada por cria_configuracao.
        gravar       True = grava as tabelas Delta de cada etapa e o resumo.

    Devolve
        {"entrada", "selecao", "automl", "calibracao", "regua", "lifetime",
         "resumo_execucao", "id_execucao_mlflow"}
        "lifetime" é None na LGD.
    """
    resultados = {"lifetime": None, "id_execucao_mlflow": None}
    linhas_resumo = []

    def registra_etapa(etapa, nome, inicio, resultado_principal):
        segundos = time.time() - inicio
        linhas_resumo.append({"etapa": etapa, "nome": nome, "tempo_s": round(segundos, 1),
                              "resultado_principal": resultado_principal})
        print(f"\n[orquestrador] etapa {etapa} ({nome}) concluída em {segundos / 60:.1f} min: "
              f"{resultado_principal}\n")

    # MLflow: uma execução para o pipeline inteiro (se houver experimento).
    usa_mlflow = cfg.get("experimento_mlflow") is not None
    if usa_mlflow:
        import mlflow
        mlflow.set_experiment(cfg["experimento_mlflow"])
        execucao = mlflow.start_run(run_name=f"{cfg['tarefa']}_{cfg['id_execucao']}")
        resultados["id_execucao_mlflow"] = execucao.info.run_id
        registra_configuracao_mlflow(cfg)

    inicio_total = time.time()
    try:
        # ---- etapa 1: entrada, validação e divisão treino/teste ------------
        inicio = time.time()
        entrada = executa_etapa_entrada(spark, sdf_entrada, cfg, gravar=gravar)
        resultados["entrada"] = entrada
        contagem = {p: int(n) for p, n in entrada["resumo_particao"].set_index("particao")["contratos"].items()}
        registra_etapa("1", "entrada", inicio,
                       f"{len(entrada['variaveis'])} variáveis candidatas; contratos por partição: {contagem}")

        # ---- etapas 2 e 3: pré-filtros e Boruta ----------------------------
        inicio = time.time()
        selecao = executa_etapa_selecao(spark, entrada["base"], entrada["variaveis"], cfg, gravar=gravar)
        resultados["selecao"] = selecao
        registra_etapa("2–3", "seleção", inicio,
                       f"{len(selecao['variaveis_selecionadas'])} variáveis selecionadas: "
                       f"{', '.join(selecao['variaveis_selecionadas'])}")
        if not selecao["variaveis_selecionadas"]:
            raise RuntimeError("Nenhuma variável passou pela seleção; o pipeline para aqui. "
                               "Veja a tabela de pré-filtros e o resultado do Boruta.")

        # ---- etapa 4: AutoML -------------------------------------------------
        inicio = time.time()
        automl = executa_etapa_automl(spark, selecao, cfg, gravar=gravar)
        resultados["automl"] = automl
        valor_oot = automl["avaliacao_final"]["oot"][cfg["metrica"]]
        registra_etapa("4", "automl", inicio,
                       f"{automl['tecnica_final']} ({automl['versao_final']}): {cfg['metrica']} OOT = {valor_oot:.4f}")
        if usa_mlflow:
            _registra_etiquetas_mlflow(automl, cfg)

        # ---- etapas 5 e 6: calibração e explicabilidade --------------------
        inicio = time.time()
        calibracao = executa_etapa_calibracao_explicabilidade(spark, automl, selecao, cfg, gravar=gravar)
        resultados["calibracao"] = calibracao
        fidelidade = calibracao["fidelidade"]
        registra_etapa("5–6", "calibração e explicabilidade", inicio,
                       f"calibração {calibracao['calibracao']['metodo']}; espelho "
                       f"{'aprovado' if fidelidade['aprovado'].all() else 'REPROVADO'} "
                       f"(Spearman mínimo {fidelidade['spearman'].min():.4f})")

        # ---- etapa 7: régua e backtest ---------------------------------------
        inicio = time.time()
        regua = executa_etapa_regua(spark, calibracao["base_calibrada"], cfg, gravar=gravar)
        resultados["regua"] = regua
        registra_etapa("7", "régua e backtest", inicio, _resumo_regua(regua, cfg))

        # ---- etapa 8: PD lifetime (só PD) ----------------------------------
        if tem_datas_de_evento(cfg):
            inicio = time.time()
            lifetime = executa_etapa_lifetime(spark, regua["base_com_rating"], regua["regua"], cfg, gravar=gravar)
            resultados["lifetime"] = lifetime
            registra_etapa("8", "PD lifetime", inicio, _resumo_lifetime(lifetime))

        # ---- resumo da execução --------------------------------------------
        resumo = pd.DataFrame(linhas_resumo)
        resumo.loc[len(resumo)] = {"etapa": "total", "nome": "pipeline",
                                   "tempo_s": round(time.time() - inicio_total, 1),
                                   "resultado_principal": f"id_execucao = {cfg['id_execucao']}"}
        resultados["resumo_execucao"] = resumo
        exibe(resumo, "Resumo da execução")
        if gravar:
            grava_tabela_delta(spark, resumo, "resumo_execucao", cfg)
        if usa_mlflow:
            mlflow.log_table(resumo, "resumo_execucao.json")
            mlflow.end_run(status="FINISHED")
    except Exception:
        # Encerra a execução do MLflow como falha e repassa o erro original.
        if usa_mlflow:
            mlflow.end_run(status="FAILED")
        raise
    return resultados

In [ ]:
from pyspark.sql import functions as F

# Lê a tabela de entrada.
sdf_entrada = spark.table(TABELA_ENTRADA)

# O pipeline trabalha com meses e representa cada mês pelo dia 01. Bases que
# guardam a data no último dia do mês (ex.: 2026-09-30) são levadas ao dia 01
# do mesmo mês (2026-09-01). Datas que já estão no dia 01 não mudam.
colunas_de_data = [cfg["colunas"]["safra"]]
if tem_datas_de_evento(cfg):
    colunas_de_data += [cfg["colunas"]["data_primeiro_default"], cfg["colunas"]["data_fim_observacao"]]
for coluna in colunas_de_data:
    sdf_entrada = sdf_entrada.withColumn(coluna, F.trunc(F.col(coluna), "month"))

# Executa o pipeline inteiro.
resultados = executa_pipeline(spark, sdf_entrada, cfg, gravar=GRAVAR)

## 4. Resultados

Cada etapa já exibiu as suas tabelas durante a execução. As células abaixo reúnem os resultados principais num lugar só. Todos os objetos ficam em `resultados[...]`; os nomes das chaves de cada etapa estão na seção "Exemplo de uso" do notebook correspondente.

### 4.1 Seleção de variáveis e AutoML

In [ ]:
exibe(resultados["selecao"]["funil"], "Funil da seleção de variáveis")
display(resultados["selecao"]["figura_boruta"])

In [ ]:
exibe(resultados["automl"]["tabela_comparativa"], "AutoML: todas as técnicas, teste e OOT")
display(resultados["automl"]["figura_comparacao"])

### 4.2 Calibração e explicabilidade

In [ ]:
exibe(resultados["calibracao"]["tabela_calibracao"], "Calibração: antes × depois")
display(resultados["calibracao"]["figura_calibracao"])
exibe(resultados["calibracao"]["fidelidade"], "Fidelidade do espelho")

In [ ]:
# No modelo de duas etapas há um par de gráficos por etapa ("etapa1" e "etapa2").
for parte, figura in resultados["calibracao"]["figuras_beeswarm"].items():
    display(figura)
    display(resultados["calibracao"]["figuras_importancia"][parte])

### 4.3 Régua e backtest

In [ ]:
exibe(resultados["regua"]["regua"], "Régua")
exibe(resultados["regua"]["registro_fusoes"], "Registro de fusões")
display(resultados["regua"]["figura_fusao"])

In [ ]:
exibe(resultados["regua"]["backtest"]["total_periodo"], "Backtest: carteira por período")
exibe(resultados["regua"]["backtest"]["rating_periodo"], "Backtest: rating por período")
display(resultados["regua"]["figura_ponte"])
display(resultados["regua"]["figura_mapa_status"])
display(resultados["regua"]["figura_ordem"])

In [ ]:
# Um gráfico por rating: observado × predito por safra, com treino, teste e OOT marcados.
for rating, figura in resultados["regua"]["figuras_rating"].items():
    display(figura)

### 4.4 PD lifetime (só PD)

In [ ]:
if resultados["lifetime"] is not None:
    lifetime = resultados["lifetime"]
    exibe(lifetime["resumo_curvas"], "PD acumulada por rating e horizonte")
    exibe(lifetime["backtest"], "Backtest da curva: prevista × observada (Kaplan-Meier)")
    exibe(lifetime["resumo_vivos"], "Contratos vivos na data de corte")
    for nome in ("mapa_safra_idade", "curvas_todos"):
        display(lifetime["figuras"][nome])
else:
    print("Etapa 8 (PD lifetime) não executada: tarefa LGD, ou base de PD sem as colunas "
          "data_primeiro_default e data_fim_observacao.")

In [ ]:
# Gráficos por rating da PD lifetime: hazard, curva, backtest e PD marginal.
# Troque a lista para ver só alguns ratings, ex.: ["A", "E", "I"].
RATINGS_PARA_EXIBIR = None   # None = todos
if resultados["lifetime"] is not None:
    figuras = resultados["lifetime"]["figuras"]
    for rating in (RATINGS_PARA_EXIBIR or list(figuras["curva"].keys())):
        for nome in ("hazard", "curva", "backtest_curva", "marginal_contrato"):
            display(figuras[nome][rating])

## 5. Onde os resultados ficam

| onde | o quê |
|---|---|
| tabelas Delta `<esquema>.<prefixo>_<tarefa>_<nome>` | cada tabela de cada etapa, com a coluna `id_execucao`; filtre por ela para ver uma execução |
| `<prefixo>_<tarefa>_resumo_execucao` | uma linha por etapa: tempo e resultado principal |
| MLflow, execução `<tarefa>_<id_execucao>` | configuração completa, versões das bibliotecas, métricas do modelo final (`final_<métrica>_<conjunto>`), etiquetas e todos os gráficos |
| `cfg["caminho_artefatos"]/<id_execucao>/` | o modelo Spark ajustado e o `modelo.json` (se `caminho_artefatos` foi definido) |
| `resultados[...]` nesta sessão | todos os objetos (DataFrames Spark, tabelas pandas, figuras) enquanto o cluster estiver ativo |